# Data analysis of results from evaluation exercise

## Import the data

In [ ]:
import pandas as pd
import numpy as np
import krippendorff
from statsmodels.stats.inter_rater import fleiss_kappa
from statsmodels.stats.inter_rater import aggregate_raters
import re
import math

In [ ]:
# Evaluation exercise data

pathname= '{insert pathname}/results_spreadsheet.xlsx'
df = pd.read_excel(pathname)
df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
df

,Proposal number,Scientific Discipline,Reviewer,Label,Type,Evaluation,Comments
0,CH-5879,Chemistry,Anna H,10247: Perovskite Materials and Applications,Predicted Topic 1,Relevant,NaN
1,CH-5879,Chemistry,Anna H,11878: Solid-state spectroscopy and crystallog...,Predicted Topic 2,Not relevant,NaN
2,CH-5879,Chemistry,Anna H,10939: Crystal Structures and Properties,Predicted Topic 3,Relevant,NaN
3,CH-5879,Chemistry,Anna H,2208: Electrical and Electronic Engineering,Predicted Subfield 1,NaN,NaN
4,CH-5879,Chemistry,Anna H,2505: Materials Chemistry,Predicted Subfield 2,NaN,NaN
...,...,...,...,...,...,...,...
5638,SC-5391,Soft Condensed Matter Science,Ellen H,1312: Molecular Biology,Predicted Subfield 2,NaN,NaN
5639,SC-5391,Soft Condensed Matter Science,Ellen H,NaN,Predicted Subfield 3,NaN,NaN
5640,SC-5391,Soft Condensed Matter Science,Ellen H,"13: Biochemistry, Genetics and Molecular Biology",Predicted Field 1,NaN,NaN
5641,SC-5391,Soft Condensed Matter Science,Ellen H,NaN,Predicted Field 2,NaN,NaN


In [ ]:
def evaluation_to_score(evaluation):
    if evaluation == 'Relevant':
        return 1
    elif evaluation == 'Unclear':
        return 0.5
    elif evaluation == 'Not relevant':
        return 0
    else:
        return np.nan

In [ ]:
df['Evaluation'] = df['Evaluation'].apply(evaluation_to_score)

## How many datapoints in total?

In [ ]:
len(df[df['Evaluation'].notna()])

2111

## Do all proposals have three unique reviewers?

In [ ]:
# Count unique reviewers per proposal
reviewer_counts = df.groupby('Proposal number')['Reviewer'].nunique().sort_values(ascending=False)

# Show counts per proposal and a small summary
reviewer_counts.head(20)  # display top 20 (change or remove .head() to see all)
print('Number of proposals with < 3 unique reviewers:', (reviewer_counts < 3).sum())
reviewer_counts.value_counts().sort_index()  # distribution of reviewer counts

Number of proposals with < 3 unique reviewers: 0


Reviewer
3    209
Name: count, dtype: int64

## Inter-rater agreement

For inter-rater agreement, we will only look at the Topic evaluations, since the sample size for Subfield and Fields evaluations are too small.

In [ ]:
target_ls = ['Predicted Topic 1', 'Predicted Topic 2', 'Predicted Topic 3']
df_topics = df[df['Type'].isin(target_ls)].copy()
df_topics = df_topics[df_topics['Evaluation'].notna()]

In [ ]:
df_topics.tail()

,Proposal number,Scientific Discipline,Reviewer,Label,Type,Evaluation,Comments
5626,SC-5384,Soft Condensed Matter Science,Ellen H,10338: Advanced Sensor and Energy Harvesting M...,Predicted Topic 2,0.0,NaN
5627,SC-5384,Soft Condensed Matter Science,Ellen H,"11799: Adhesion, Friction, and Surface Interac...",Predicted Topic 3,1.0,NaN
5634,SC-5391,Soft Condensed Matter Science,Ellen H,10492: Microtubule and mitosis dynamics,Predicted Topic 1,1.0,NaN
5635,SC-5391,Soft Condensed Matter Science,Ellen H,10303: Photosynthetic Processes and Mechanisms,Predicted Topic 2,0.0,NaN
5636,SC-5391,Soft Condensed Matter Science,Ellen H,13526: 14-3-3 protein interactions,Predicted Topic 3,0.5,NaN


### Krippendorf's alpha

We need to pivot the DataFrame into a reliability data matrix (RDM).

In [ ]:
# Create 'reviewer order' column that is needed for pivoting the dataframe into the reliability data matrix
df_topics['reviewer order'] = df_topics.groupby(['Proposal number', 'Label']).cumcount()

In [ ]:
rdm_filtered = df_topics.pivot_table(index=['Proposal number', 'Label','Scientific Discipline'], columns='reviewer order', values='Evaluation').reset_index()

In [ ]:
not_others_ls = ['Chemistry', 'Applied Material Science', 'Hard Condensed Matter Science', 'Soft Condensed Matter Science']

print('Krippendorff\'s alpha for Chemistry:',krippendorff.alpha(reliability_data=rdm_filtered[rdm_filtered['Scientific Discipline'] == 'Chemistry'][[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))
print('Krippendorff\'s alpha for Applied Material Science:',krippendorff.alpha(reliability_data=rdm_filtered[rdm_filtered['Scientific Discipline'] == 'Applied Material Science'][[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))
print('Krippendorff\'s alpha for Hard Condensed Matter Science:',krippendorff.alpha(reliability_data=rdm_filtered[rdm_filtered['Scientific Discipline'] == 'Hard Condensed Matter Science'][[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))
print('Krippendorff\'s alpha for Soft Condensed Matter Science:',krippendorff.alpha(reliability_data=rdm_filtered[rdm_filtered['Scientific Discipline'] == 'Soft Condensed Matter Science'][[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))
print('Krippendorff\'s alpha for Others:',krippendorff.alpha(reliability_data=rdm_filtered[~rdm_filtered['Scientific Discipline'].isin(not_others_ls)][[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))
print('Krippendorff\'s alpha for All:',krippendorff.alpha(reliability_data=rdm_filtered[[0,1,2]].to_numpy().T, level_of_measurement='ordinal'))

Krippendorff's alpha for Chemistry: 0.5739944072320082
Krippendorff's alpha for Applied Material Science: 0.49722012809934035
Krippendorff's alpha for Hard Condensed Matter Science: 0.6160512947846178
Krippendorff's alpha for Soft Condensed Matter Science: 0.6137139153359238
Krippendorff's alpha for Others: 0.5133936137087682
Krippendorff's alpha for All: 0.5719933490424536


### Fleiss' and Randolph's kappas

We calculate the Fleiss' and Randolph's kappas, which is another inter-rater agreement metric.

In [ ]:
# Function to print Fleiss' kappa and Randolph's kappa for a given discipline
def print_kappas(rdm,discipline):

    not_others_ls = ['Chemistry', 'Applied Material Science', 'Hard Condensed Matter Science', 'Soft Condensed Matter Science']
    if discipline == 'All':
        table, categories = aggregate_raters(rdm[[0,1,2]])     # Create the table needed for Fleiss' kappa calculation

    elif discipline == 'Others':
        rdm = rdm[~rdm['Scientific Discipline'].isin(not_others_ls)]
        table, categories = aggregate_raters(rdm[[0,1,2]])     # Create the table needed for Fleiss' kappa calculation

    else:
        table, categories = aggregate_raters(rdm[rdm['Scientific Discipline'] == discipline][[0,1,2]])   # Create the table needed for Fleiss' kappa calculation for a particular discipline
    k = fleiss_kappa(table, method='fleiss')
    r = fleiss_kappa(table, method='randolph')

    print(f"Fleiss' Kappa for {discipline}:", k)
    print(f"Randolph's Kappa for {discipline}:", r)


In [ ]:
print_kappas(rdm_filtered,'Chemistry')
print_kappas(rdm_filtered,'Applied Material Science')
print_kappas(rdm_filtered,'Hard Condensed Matter Science')
print_kappas(rdm_filtered,'Soft Condensed Matter Science')
print_kappas(rdm_filtered,'Others')
print_kappas(rdm_filtered,'All')

Fleiss' Kappa for Chemistry: 0.4627551020408164
Randolph's Kappa for Chemistry: 0.5465116279069767
Fleiss' Kappa for Applied Material Science: 0.3905027345393355
Randolph's Kappa for Applied Material Science: 0.5141843971631207
Fleiss' Kappa for Hard Condensed Matter Science: 0.5113467166401396
Randolph's Kappa for Hard Condensed Matter Science: 0.6518518518518519
Fleiss' Kappa for Soft Condensed Matter Science: 0.417571546885065
Randolph's Kappa for Soft Condensed Matter Science: 0.44252873563218403
Fleiss' Kappa for Others: 0.3808399219632407
Randolph's Kappa for Others: 0.4402985074626866
Fleiss' Kappa for All: 0.4453324252764129
Randolph's Kappa for All: 0.5247603833865814


## Accuracy

In [ ]:
target_ls = ['Predicted Topic 1', 'Predicted Topic 2', 'Predicted Topic 3']
df_topics = df[df['Type'].isin(target_ls)].copy()
df_topics = df_topics[df_topics['Evaluation'].notna()]

In [ ]:
df_topics.tail()

,Proposal number,Scientific Discipline,Reviewer,Label,Type,Evaluation,Comments
5626,SC-5384,Soft Condensed Matter Science,Ellen H,10338: Advanced Sensor and Energy Harvesting M...,Predicted Topic 2,0.0,NaN
5627,SC-5384,Soft Condensed Matter Science,Ellen H,"11799: Adhesion, Friction, and Surface Interac...",Predicted Topic 3,1.0,NaN
5634,SC-5391,Soft Condensed Matter Science,Ellen H,10492: Microtubule and mitosis dynamics,Predicted Topic 1,1.0,NaN
5635,SC-5391,Soft Condensed Matter Science,Ellen H,10303: Photosynthetic Processes and Mechanisms,Predicted Topic 2,0.0,NaN
5636,SC-5391,Soft Condensed Matter Science,Ellen H,13526: 14-3-3 protein interactions,Predicted Topic 3,0.5,NaN


### How many proposals have less than three Topic predictions?

In [ ]:
count = df_topics.groupby('Proposal number')['Label'].nunique()
print('Number of proposals with less than 3 Topics:',len(count[count != 3]))
print('Proposals with less than 3 Topics:',count[count != 3].index.tolist())

Number of proposals with less than 3 Topics: 1
Proposals with less than 3 Topics: ['EV-514']


In [ ]:
df[df['Proposal number']=='EV-514'].tail(9)

,Proposal number,Scientific Discipline,Reviewer,Label,Type,Evaluation,Comments
4257,EV-514,Environment,Miguel G,12012: Diatoms and Algae Research,Predicted Topic 1,0.0,The cyanobacterias are not diatom or algae in ...
4258,EV-514,Environment,Miguel G,11740: Geochemistry and Elemental Analysis,Predicted Topic 2,0.0,"Likewise, while elemental analysis may point t..."
4259,EV-514,Environment,Miguel G,NaN,Predicted Topic 3,NaN,NaN
4260,EV-514,Environment,Miguel G,2502: Biomaterials,Predicted Subfield 1,1.0,"Regarding the predicted subfields, I think bio..."
4261,EV-514,Environment,Miguel G,1906: Geochemistry and Petrology,Predicted Subfield 2,0.0,NaN
4262,EV-514,Environment,Miguel G,NaN,Predicted Subfield 3,NaN,NaN
4263,EV-514,Environment,Miguel G,25: Materials Science,Predicted Field 1,NaN,NaN
4264,EV-514,Environment,Miguel G,19: Earth and Planetary Sciences,Predicted Field 2,NaN,NaN
4265,EV-514,Environment,Miguel G,NaN,Predicted Field 3,NaN,NaN


### Number/proportion of proposals with at least n Topics with a mean score >= x

#### x=0.667

In [ ]:
# Create new column with the 'Other' category for Scientific Discipline
not_others_ls = ['Chemistry', 'Applied Material Science', 'Hard Condensed Matter Science', 'Soft Condensed Matter Science']
df_topics['Category'] = df_topics['Scientific Discipline'].apply(lambda x: x if x in not_others_ls else 'Others')

# Calculate mean evaluation per Proposal number, Topic, and Discipline
df_mean_grouped = df_topics.groupby(['Proposal number','Label','Category'],as_index=False)['Evaluation'].mean()

# Filter the dataframe for Topics with mean evaluation > 0.5 and get the counts per proposal per category
topic_count_grouped = df_mean_grouped[df_mean_grouped['Evaluation'] > 0.5].groupby(['Proposal number','Category'],as_index=False).size()

# Calculate counts per category
grouped1 = topic_count_grouped[topic_count_grouped['size'] >= 1]['Category'].value_counts()
grouped2 = topic_count_grouped[topic_count_grouped['size'] >= 2]['Category'].value_counts()
grouped3 = topic_count_grouped[topic_count_grouped['size'] == 3]['Category'].value_counts()

# Total proposals per category
total = df_mean_grouped[['Proposal number','Category']].groupby('Category').nunique()['Proposal number']

# Create dataframe with counts
df_grouped_count = pd.DataFrame({'At least 1 Topic':grouped1, 'At least 2 Topics':grouped2, '3 Topics':grouped3, 'Total proposals':total}).reset_index()

# Fill NaN values with 0
df_grouped_count.fillna(0,inplace=True)

# Create a 'Total' category
df_grouped_count.loc[len(df_grouped_count)] = ['Total'] + df_grouped_count.sum(numeric_only=True).tolist()

# Calculate proportions
df_grouped_count['Proportion with at least 1 Topic'] = df_grouped_count['At least 1 Topic'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with at least 2 Topics'] = df_grouped_count['At least 2 Topics'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with 3 Topics'] = df_grouped_count['3 Topics'] / df_grouped_count['Total proposals']

df_grouped_count

,Category,At least 1 Topic,At least 2 Topics,3 Topics,Total proposals,Proportion with at least 1 Topic,Proportion with at least 2 Topics,Proportion with 3 Topics
0,Applied Material Science,46,31,11,47,0.978723,0.659574,0.234043
1,Chemistry,38,22,6,43,0.883721,0.511628,0.139535
2,Hard Condensed Matter Science,44,36,12,45,0.977778,0.800000,0.266667
3,Others,38,21,3,45,0.844444,0.466667,0.066667
4,Soft Condensed Matter Science,26,16,1,29,0.896552,0.551724,0.034483
5,Total,192,126,33,209,0.918660,0.602871,0.157895


#### x = 0.883

In [ ]:
# Create new column with the 'Other' category for Scientific Discipline
not_others_ls = ['Chemistry', 'Applied Material Science', 'Hard Condensed Matter Science', 'Soft Condensed Matter Science']
df_topics['Category'] = df_topics['Scientific Discipline'].apply(lambda x: x if x in not_others_ls else 'Others')

# Calculate mean evaluation per Proposal number, Topic, and Discipline
df_mean_grouped = df_topics.groupby(['Proposal number','Label','Category'],as_index=False)['Evaluation'].mean()

# Filter the dataframe for Topics with mean evaluation > 0.8 and get the counts per proposal per category
topic_count_grouped = df_mean_grouped[df_mean_grouped['Evaluation'] > 0.8].groupby(['Proposal number','Category'],as_index=False).size()

# Calculate counts per category
grouped1 = topic_count_grouped[topic_count_grouped['size'] >= 1]['Category'].value_counts()
grouped2 = topic_count_grouped[topic_count_grouped['size'] >= 2]['Category'].value_counts()
grouped3 = topic_count_grouped[topic_count_grouped['size'] == 3]['Category'].value_counts()

# Total proposals per category
total = df_mean_grouped[['Proposal number','Category']].groupby('Category').nunique()['Proposal number']

# Create dataframe with counts
df_grouped_count = pd.DataFrame({'At least 1 Topic':grouped1, 'At least 2 Topics':grouped2, '3 Topics':grouped3, 'Total proposals':total}).reset_index()

# Fill NaN values with 0
df_grouped_count.fillna(0,inplace=True)

# Create a 'Total' category
df_grouped_count.loc[len(df_grouped_count)] = ['Total'] + df_grouped_count.sum(numeric_only=True).tolist()

# Calculate proportions
df_grouped_count['Proportion with at least 1 Topic'] = df_grouped_count['At least 1 Topic'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with at least 2 Topics'] = df_grouped_count['At least 2 Topics'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with 3 Topics'] = df_grouped_count['3 Topics'] / df_grouped_count['Total proposals']

df_grouped_count

,Category,At least 1 Topic,At least 2 Topics,3 Topics,Total proposals,Proportion with at least 1 Topic,Proportion with at least 2 Topics,Proportion with 3 Topics
0,Applied Material Science,41,21,6,47,0.872340,0.446809,0.127660
1,Chemistry,34,13,1,43,0.790698,0.302326,0.023256
2,Hard Condensed Matter Science,43,32,6,45,0.955556,0.711111,0.133333
3,Others,32,10,1,45,0.711111,0.222222,0.022222
4,Soft Condensed Matter Science,23,10,1,29,0.793103,0.344828,0.034483
5,Total,173,86,15,209,0.827751,0.411483,0.071770


#### x = 1

In [ ]:
# Create new column with the 'Other' category for Scientific Discipline
not_others_ls = ['Chemistry', 'Applied Material Science', 'Hard Condensed Matter Science', 'Soft Condensed Matter Science']
df_topics['Category'] = df_topics['Scientific Discipline'].apply(lambda x: x if x in not_others_ls else 'Others')

# Calculate mean evaluation per Proposal number, Topic, and Discipline
df_mean_grouped = df_topics.groupby(['Proposal number','Label','Category'],as_index=False)['Evaluation'].mean()

# Filter the dataframe for Topics with mean evaluation = 1 and get the counts per proposal per category
topic_count_grouped = df_mean_grouped[df_mean_grouped['Evaluation'] == 1].groupby(['Proposal number','Category'],as_index=False).size()

# Calculate counts per category
grouped1 = topic_count_grouped[topic_count_grouped['size'] >= 1]['Category'].value_counts()
grouped2 = topic_count_grouped[topic_count_grouped['size'] >= 2]['Category'].value_counts()
grouped3 = topic_count_grouped[topic_count_grouped['size'] == 3]['Category'].value_counts()

# Total proposals per category
total = df_mean_grouped[['Proposal number','Category']].groupby('Category').nunique()['Proposal number']

# Create dataframe with counts
df_grouped_count = pd.DataFrame({'At least 1 Topic':grouped1, 'At least 2 Topics':grouped2, '3 Topics':grouped3, 'Total proposals':total}).reset_index()

# Fill NaN values with 0
df_grouped_count.fillna(0,inplace=True)

# Create a 'Total' category
df_grouped_count.loc[len(df_grouped_count)] = ['Total'] + df_grouped_count.sum(numeric_only=True).tolist()

# Calculate proportions
df_grouped_count['Proportion with at least 1 Topic'] = df_grouped_count['At least 1 Topic'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with at least 2 Topics'] = df_grouped_count['At least 2 Topics'] / df_grouped_count['Total proposals']
df_grouped_count['Proportion with 3 Topics'] = df_grouped_count['3 Topics'] / df_grouped_count['Total proposals']

df_grouped_count

,Category,At least 1 Topic,At least 2 Topics,3 Topics,Total proposals,Proportion with at least 1 Topic,Proportion with at least 2 Topics,Proportion with 3 Topics
0,Applied Material Science,39.0,15.0,1.0,47.0,0.829787,0.319149,0.021277
1,Chemistry,31.0,10.0,0.0,43.0,0.720930,0.232558,0.000000
2,Hard Condensed Matter Science,38.0,29.0,4.0,45.0,0.844444,0.644444,0.088889
3,Others,28.0,4.0,0.0,45.0,0.622222,0.088889,0.000000
4,Soft Condensed Matter Science,19.0,6.0,0.0,29.0,0.655172,0.206897,0.000000
5,Total,155.0,64.0,5.0,209.0,0.741627,0.306220,0.023923


### In cases where all Topics are 'Unclear'/'Not relevant', are the Subfields good enough?

In [ ]:
target_ls = ['Predicted Subfield 1', 'Predicted Subfield 2', 'Predicted Subfield 3']
df_subfields_all = df[df['Type'].isin(target_ls)].copy()
df_subfields = df_subfields_all[df_subfields_all['Evaluation'].notna()]

print('Number of proposals where at least one reviewers has marked all Topics as Unclear/Not relevant:',df_subfields['Proposal number'].nunique())
print('Number of instances where a reviewer has marked all Topics as Unclear/Not relevant:', len(df_subfields.groupby(['Proposal number','Reviewer'])))
print('Number of instances where at least one Subfield is subsequently marked as Relevant:',len(df_subfields[df_subfields['Evaluation']==1].groupby(['Proposal number','Reviewer'])))

Number of proposals where at least one reviewers has marked all Topics as Unclear/Not relevant: 49
Number of instances where a reviewer has marked all Topics as Unclear/Not relevant: 73
Number of instances where at least one Subfield is subsequently marked as Relevant: 55


### In cases where all Subfields are 'Unclear'/'Not relevant', are the Fields good enough?

In [ ]:
target_ls = ['Predicted Field 1', 'Predicted Field 2', 'Predicted Field 3']
df_fields_all = df[df['Type'].isin(target_ls)].copy()
df_fields = df_fields_all[df_fields_all['Evaluation'].notna()]

print('Number of proposals where at least one reviewers has marked all Subfields as Unclear/Not relevant:',df_fields['Proposal number'].nunique())
print('Number of instances where a reviewer has marked all Subfields as Unclear/Not relevant:', len(df_fields.groupby(['Proposal number','Reviewer'])))
print('Number of instances where at least one Field is subsequently marked as Relevant:',len(df_fields[df_fields['Evaluation']==1].groupby(['Proposal number','Reviewer'])))

Number of proposals where at least one reviewers has marked all Subfields as Unclear/Not relevant: 16
Number of instances where a reviewer has marked all Subfields as Unclear/Not relevant: 18
Number of instances where at least one Field is subsequently marked as Relevant: 8


### Why does the model fail completely in some cases?

In [ ]:
df_subfields['Proposal number'].unique()

array(['CH-5900', 'CH-6138', 'CH-6143', 'CH-6158', 'CH-6162', 'CH-6290',
       'CH-6424', 'CH-6556', 'CH-6640', 'EV-514', 'HC-4794', 'LS-2968',
       'LS-3092', 'LS-3131', 'MA-4840', 'MA-5014', 'MA-5201', 'MA-5576',
       'MX-2415', 'MX-2422', 'MX-2510', 'MX-2513', 'SC-5105', 'SC-5110',
       'SC-5141', 'SC-5189', 'SC-5277', 'SC-5298', 'SC-5302', 'SC-5369',
       'SC-5384', 'CH-6361', 'CH-6492', 'ES-1118', 'ES-1119', 'HC-4882',
       'LS-3034', 'MA-5204', 'MA-5779', 'MI-1429', 'MX-2333', 'MX-2443',
       'SC-5247', 'CH-6381', 'ES-1032', 'HC-4565', 'HC-4689', 'LS-3012',
       'LS-3157'], dtype=object)

In [ ]:
df_subfields[df_subfields['Comments'].notna()].tail()

,Proposal number,Scientific Discipline,Reviewer,Label,Type,Evaluation,Comments
4350,HC-4565,Hard Condensed Matter Science,Oliver C,"2105: Renewable Energy, Sustainability and the...",Predicted Subfield 1,0.0,This is a characterisation of electronic prope...
4351,HC-4565,Hard Condensed Matter Science,Oliver C,2502: Biomaterials,Predicted Subfield 2,0.0,This is a characterisation of electronic prope...
4352,HC-4565,Hard Condensed Matter Science,Oliver C,2312: Water Science and Technology,Predicted Subfield 3,0.0,This is a characterisation of electronic prope...
5313,MI-1429,Method and Instrumentation,Oliver C,1908: Geophysics,Predicted Subfield 1,0.0,This is an extension of the proposal two rows ...
5315,MI-1429,Method and Instrumentation,Oliver C,2204: Biomedical Engineering,Predicted Subfield 3,0.0,This is an extension of the proposal two rows ...


In [ ]:
df_subfields[df_subfields['Proposal number']=='LS-3131']['Comments'].iloc[0]

'Metal localization after antibiotic treatment in Gram negative bacteria'

### Compare expert ratings to model confidence

In [ ]:
# Import model predictions for the 209 proposals from JSON file
# Evaluation exercise data
file_path = '{insert pathname}/proposals_for_validation.json'
model_pred = pd.read_json(file_path)

In [ ]:
# Create a function to extract Topic prediction IDs from the model predictions dataframe
def extract_model_predictions_ids(row):
    ls = []
    for item in row:
        ls.append(item['topic_id'])
    return ls

# Create a function to extract model confidence from the model predictions dataframe
def extract_model_predictions_scores(row):
    ls = []
    for item in row:
        ls.append(item['topic_score'])
    return ls


In [ ]:
# Explode the lists of IDss and scores into separate rows
ids = model_pred['topic_predictions_pdf_metadata_only'].apply(extract_model_predictions_ids).explode()
scores = model_pred['topic_predictions_pdf_metadata_only'].apply(extract_model_predictions_scores).explode()

# Rename the series for clarity
ids.rename('ID', inplace=True)
scores.rename('Model confidence', inplace=True)

# Concatenate IDs and scores into a single dataframe
merge_pred = pd.concat([ids, scores], axis=1)

# Merge with proposal numbers based on index
model_confidence = pd.merge(model_pred['proposal_number'], merge_pred, left_index=True, right_index=True)

# Rename 'proposal_number' column to 'Proposal number' for merging later
model_confidence.rename(columns={'proposal_number':'Proposal number'}, inplace=True)

In [ ]:
# Function to extract the ID from df_mean_grouped labels for merging later
def extract_ID(row):
    match = re.search(r'\d+', row)
    number = match.group()
    number = int(number)
    return number

In [ ]:
# Apply the function to create 'ID' column in df_mean_grouped
df_mean_grouped['ID'] = df_mean_grouped['Label'].apply(extract_ID)

# Merge the dataframes
df_final = pd.merge(df_mean_grouped, model_confidence, left_on=['Proposal number','ID'], right_on=['Proposal number','ID'])

In [ ]:
mse = ((df_final['Evaluation'] - df_final['Model confidence'])**2).mean()
rmse = math.sqrt(mse)

print('RMSE between human evaluation and model confidence:', rmse)

RMSE between human evaluation and model confidence: 0.5647004790570072


## How many Topic predictions in total?

In [ ]:
len(df_topics.groupby(['Proposal number','Reviewer']))

627